# Story Reader — specialist server (Colab GPU)

This notebook runs TWO models on a free Colab GPU behind ONE public URL:
- the **football detector** (Roboflow `football-players-detection-3zvbc/11`: players, goalkeepers, referees, ball)
  for MVP 3's real live test: 3 cameras x 5 frames per second while the match plays (the Mac is too slow for that);
- the **ball action spotting** model
[`OpenSportsLab/OSL-loc-snbas-2025-e2e`](https://huggingface.co/OpenSportsLab/OSL-loc-snbas-2025-e2e)
(12 classes: PASS, DRIVE, HEADER, HIGH PASS, OUT, CROSS, THROW IN, SHOT, BALL PLAYER BLOCK,
PLAYER SUCCESSFUL TACKLE, FREE KICK, GOAL) on a free Colab GPU, and gives you a public URL that the
dashboard on your Mac connects to.

## How to run (5 steps)
1. **Turn on the GPU:** menu *Runtime → Change runtime type → T4 GPU → Save*.
2. **Roboflow key (for the detector):** click the 🔑 *Secrets* icon (left bar) → *Add new secret* → name
   `ROBOFLOW_API_KEY` (exactly), value = your Roboflow API key, switch *Notebook access* on. Without it the
   action model still runs, only the detector is off.
3. *(Optional, only for the action model alone; the detector needs far more than ngrok's free 1 GB/month)* **ngrok token:** make a free account at https://dashboard.ngrok.com, copy your
   authtoken, then in Colab click the 🔑 *Secrets* icon (left bar) → *Add new secret* → name
   `NGROK_AUTHTOKEN`, value = your token, and switch *Notebook access* on.
   By default the notebook uses a free Cloudflare quick tunnel (no account, no data cap); ngrok only with TUNNEL = 'ngrok' in step 5.
4. **Run everything:** menu *Runtime → Run all*. The first run takes ~5–8 minutes (installing + downloading the models).
5. **Copy the URL** printed in big letters at the end (it looks like `https://xxxx.ngrok-free.app`
   or `https://xxxx.trycloudflare.com`).
6. **Paste it** into the dashboard: MVP 3 http://localhost:8000/studio → ● Live → *Colab URL* → *Test*
   (both the action model and the detector should show), or MVP 2 http://localhost:8000/story → *Connect*.

Keep this tab open while you use the dashboard. Colab stops the server after ~90 minutes of no activity
(free tier); then just *Run all* again and paste the new URL.

**License note:** the model is AGPL-3.0 (research / non-commercial use).

## 1. Check the GPU and install packages

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo 'NO GPU: Runtime -> Change runtime type -> T4 GPU'
!pip install -q opensportslib==0.3.1 fastapi uvicorn pyngrok websockets huggingface_hub pyyaml
import torch; print('CUDA available:', torch.cuda.is_available())

## 2. The server code (same as `colab/specialist_server.py` in the project)

In [ ]:
%%writefile specialist_server.py
"""specialist_server.py — the SPECIALIST layer server (runs on a Colab GPU).

This file is the source of the server cells in colab/specialist_server.ipynb
(colab/build_notebook.py copies it into the notebook, so the notebook stays self-contained).
It can also run locally for testing:  python colab/specialist_server.py --port 8765   (CPU = slow)

Model: OpenSportsLab/OSL-loc-snbas-2025-e2e (OpenSportsLib, E2E-Spot: RegNetY-008 + GSM + GRU),
trained on SoccerNet Ball Action Spotting 2025, 12 classes. From its config.yaml:
  - frames: 224 x 398 (H x W), RGB, /255, ImageNet mean/std
  - sampling: input 25 fps, stride int(25 / extract_fps=2) = 12  ->  one frame every 0.48 s (~2.08 fps)
  - clip_len: 100 frames  ->  each forward pass sees the last ~48 s
  - output: per frame, softmax over 13 = background + 12 classes

API:
  GET /health   model, classes, expected fps, frame size, window length, GPU name
  WS  /stream   client sends binary messages: 8-byte little-endian float64 timestamp (s) + JPEG bytes.
                Every `stride_s` the server runs the model on the latest window (PAST frames only:
                it only has what was sent) and replies:
                {"type": "events", "events": [{"class", "event_time", "confidence"}], "infer_ms", "rtf", ...}
  WS  /detect   relayed to the football DETECTOR (colab/detector_server.py, its own Python environment on port
                DETECTOR_PORT, default 8766), so one tunnel URL serves both. /health shows it under "detector".

Live detail: the model's GRU looks both ways in time, so the newest frames (no "future" yet) are
unreliable. A frame is only reported once it is `report_delay_s` old (default 2 s): a small,
honest delay instead of many false alarms.
"""
from __future__ import annotations

import argparse
import asyncio
import json
import os
import struct
import time
from collections import deque

import cv2
import numpy as np
from fastapi import FastAPI, WebSocket, WebSocketDisconnect

DETECTOR_PORT = int(os.environ.get("DETECTOR_PORT", "8766"))
MODEL_REPO = "OpenSportsLab/OSL-loc-snbas-2025-e2e"
CLASSES = ["PASS", "DRIVE", "HEADER", "HIGH PASS", "OUT", "CROSS", "THROW IN", "SHOT",
           "BALL PLAYER BLOCK", "PLAYER SUCCESSFUL TACKLE", "FREE KICK", "GOAL"]
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)


class Spotter:
    """Loads the model once; turns a list of frames into per-frame class probabilities."""

    def __init__(self, weights_dir: str | None = None):
        import torch
        import yaml
        from huggingface_hub import hf_hub_download
        from opensportslib.apis import LocalizationModel

        self.torch = torch
        config_path = os.path.join(weights_dir, "config.yaml") if weights_dir else \
            hf_hub_download(MODEL_REPO, "config.yaml")
        weights_path = os.path.join(weights_dir, "model.pt") if weights_dir else \
            hf_hub_download(MODEL_REPO, "model.pt")
        data = yaml.safe_load(open(config_path))["DATA"]
        self.classes = list(data["classes"])
        self.height, self.width = int(data["target_height"]), int(data["target_width"])
        self.clip_len = int(data["clip_len"])
        self.stride = max(1, int(data["input_fps"] / data["extract_fps"]))
        self.fps = data["input_fps"] / self.stride
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.gpu = torch.cuda.get_device_name(0) if self.device == "cuda" else "none (CPU)"
        self.model = LocalizationModel(config=config_path, weights=weights_path).model   # E2EModel

    def health(self) -> dict:
        return {"model": MODEL_REPO, "classes": self.classes, "fps": round(self.fps, 4),
                "frame_height": self.height, "frame_width": self.width, "clip_len": self.clip_len,
                "window_s": round(self.clip_len / self.fps, 1), "device": self.device, "gpu": self.gpu}

    def prepare(self, jpeg: bytes) -> np.ndarray:
        """JPEG -> normalized CHW float32 array, exactly like the training loader (OpenCV path)."""
        frame = cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)
        if frame.shape[0] != self.height or frame.shape[1] != self.width:
            frame = cv2.resize(frame, (self.width, self.height))
        rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        return ((rgb - MEAN) / STD).transpose(2, 0, 1)

    def predict(self, frames: list[np.ndarray]) -> np.ndarray:
        """frames (L x 3 x H x W) -> probabilities (L x 13). Short windows are padded at the start."""
        L = len(frames)
        if L < self.clip_len:   # pad with copies of the first frame (the model expects clip_len frames)
            frames = [frames[0]] * (self.clip_len - L) + list(frames)
        x = self.torch.from_numpy(np.stack(frames))
        _, prob = self.model.predict(x, use_amp=self.device == "cuda")
        return prob[0][-L:]

    def warm_up(self) -> float:
        start = time.time()
        self.predict([np.zeros((3, self.height, self.width), np.float32)] * self.clip_len)
        return time.time() - start


def find_peaks(times: list[float], prob: np.ndarray, classes: list[str], start: float, end: float,
               min_confidence: float, nms_s: float) -> list[dict]:
    """Events = frames in (start, end] where a class is >= min_confidence and the highest within +-nms_s."""
    events = []
    t = np.asarray(times)
    for c, name in enumerate(classes, start=1):          # column 0 = background
        p = prob[:, c]
        for i in np.where((t > start) & (t <= end) & (p >= min_confidence))[0]:
            near = np.abs(t - t[i]) <= nms_s
            if p[i] >= p[near].max():
                events.append({"class": name, "event_time": round(float(t[i]), 2), "confidence": round(float(p[i]), 3)})
    return events


class Session:
    """One connected client: a ring buffer of the latest frames + inference every stride_s."""

    def __init__(self, spotter: Spotter, settings: dict):
        self.spotter = spotter
        self.min_confidence = float(settings.get("min_confidence", 0.3))
        self.stride_s = float(settings.get("stride_s", 2.0))
        self.report_delay_s = float(settings.get("report_delay_s", 2.0))
        self.nms_s = float(settings.get("nms_s", 1.0))
        self.frames: deque[tuple[float, np.ndarray]] = deque(maxlen=spotter.clip_len)
        self.last_infer = -1e9
        self.reported_until = -1e9
        self.busy = False

    def add(self, t: float, jpeg: bytes) -> bool:
        """Store a frame. Returns True when an inference is due."""
        if self.frames and t <= self.frames[-1][0]:
            return False                    # out of order / duplicate: ignore
        self.frames.append((t, self.spotter.prepare(jpeg)))
        return not self.busy and t - self.last_infer >= self.stride_s

    def infer(self) -> dict:
        times = [t for t, _ in self.frames]
        start = time.time()
        prob = self.spotter.predict([f for _, f in self.frames])
        infer_s = time.time() - start
        now = times[-1]
        self.last_infer = now
        until = now - self.report_delay_s
        events = find_peaks(times, prob, self.spotter.classes, self.reported_until, until,
                            self.min_confidence, self.nms_s)
        self.reported_until = max(self.reported_until, until)
        return {"type": "events", "events": events, "infer_ms": round(infer_s * 1000, 1),
                "rtf": round(infer_s / self.stride_s, 3), "window": [round(times[0], 2), round(now, 2)],
                "frames": len(times), "reported_until": round(until, 2)}


def detector_health() -> dict:
    """The detector process's /health ({"ready": False} while it is not running or still loading)."""
    import urllib.request

    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{DETECTOR_PORT}/health", timeout=1) as r:
            return json.loads(r.read())
    except Exception as error:                       # noqa: BLE001 — shown on the dashboard
        return {"ready": False, "error": type(error).__name__}


def make_app(spotter: Spotter | None):
    app = FastAPI(title="Story Reader specialist")

    @app.get("/health")
    def health() -> dict:
        return {**(spotter.health() if spotter else {"model": "none", "fps": 0}), "detector": detector_health()}

    @app.websocket("/detect")
    async def detect(ws: WebSocket) -> None:
        """Relay to the detector process: frames in, detections back (both directions at once)."""
        import websockets

        await ws.accept()
        try:
            async with websockets.connect(f"ws://127.0.0.1:{DETECTOR_PORT}/detect", max_size=2 ** 24) as inner:
                async def back() -> None:
                    async for reply in inner:
                        await ws.send_text(reply if isinstance(reply, str) else reply.decode())

                task = asyncio.create_task(back())
                try:
                    while True:
                        await inner.send(await ws.receive_bytes())
                finally:
                    task.cancel()
        except WebSocketDisconnect:
            pass
        except Exception as error:                   # noqa: BLE001 — the detector is not running
            await ws.send_json({"type": "error", "message": f"detector: {type(error).__name__}: {error}"})
            await ws.close()

    @app.websocket("/stream")
    async def stream(ws: WebSocket) -> None:
        await ws.accept()
        session = Session(spotter, {})
        loop = asyncio.get_running_loop()
        try:
            while True:
                message = await ws.receive()
                if message.get("type") == "websocket.disconnect":
                    break
                if message.get("text"):          # settings: {"min_confidence": .., "stride_s": .., ...}
                    session = Session(spotter, json.loads(message["text"]))
                    await ws.send_json({"type": "ready", **spotter.health()})
                    continue
                data = message.get("bytes") or b""
                if len(data) <= 8:
                    continue
                t = struct.unpack("<d", data[:8])[0]
                if session.add(t, data[8:]):
                    # Run the model in a thread so frames keep arriving meanwhile. While it is busy,
                    # new frames just go into the buffer (no backlog: like a live system).
                    session.busy = True

                    async def run() -> None:
                        try:
                            result = await loop.run_in_executor(None, session.infer)
                            await ws.send_json(result)
                        except Exception as error:
                            await ws.send_json({"type": "error", "message": f"{type(error).__name__}: {error}"})
                        finally:
                            session.busy = False

                    asyncio.create_task(run())
        except WebSocketDisconnect:
            pass

    return app


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--port", type=int, default=8765)
    parser.add_argument("--weights-dir", default=None, help="folder with config.yaml + model.pt (default: download)")
    parser.add_argument("--detector-only", action="store_true", help="no action model (test the /detect relay)")
    args = parser.parse_args()
    import uvicorn

    spotter = None if args.detector_only else Spotter(args.weights_dir)
    if spotter:
        print("warm-up:", round(spotter.warm_up(), 2), "s |", spotter.health())
    uvicorn.run(make_app(spotter), host="127.0.0.1", port=args.port, log_level="warning")


## 2b. The football detector in its own Python environment
Roboflow's `inference-gpu` pins many packages; in its own environment it can't break the action model's torch. It runs as a second local server (port 8766); the main server relays `/detect` to it.

In [ ]:
!pip install -q uv
!uv venv -q --seed --python 3.11 /content/detenv
# onnxruntime-gpu[cuda,cudnn]: the CUDA + cuDNN versions this onnxruntime was built for, in its own environment
!uv pip install -q --python /content/detenv/bin/python inference-gpu 'onnxruntime-gpu[cuda,cudnn]' fastapi uvicorn websockets
!/content/detenv/bin/python -c "import onnxruntime as o; print('detector env OK, onnxruntime', o.__version__, o.get_available_providers())"

In [ ]:
%%writefile detector_server.py
"""detector_server.py — the football DETECTOR on a Colab GPU (MVP 3 live test: detection while the match plays).

The Mac needs ~0.4 s per camera frame; live needs 3 cameras x 5 frames/s = 15 frames/s. A T4 GPU does it.

It runs in its OWN Python environment (the notebook makes one with uv): Roboflow's `inference-gpu` pins many
packages and must not break the action model's torch next to it. The main server (specialist_server.py) relays
WS /detect to this process, so one tunnel URL serves both.
It can also run on the Mac for testing:  venv/bin/python colab/detector_server.py --port 8766  (CoreML, slow)

Same output as the Mac's `src/detect.Detector.detect` (the live director must see what the cached runs saw):
  - the frame is padded to 16:9 on a black canvas (pad_to_aspect), the model runs at its 1280 input;
  - players = player / goalkeeper / referee >= 0.3; ball = the single best ball >= 0.3;
  - boxes in the ORIGINAL picture's pixels, rounded to 0.1 px.

API:
  GET /health   {"model", "ready", "ms_per_frame", "providers"}
  WS  /detect   client sends binary: 4-byte little-endian header length + JSON header + JPEG bytes.
                header {"frame": int, "sent": float, "items": [{"cam": int, "size": int}, ...]} = several cameras of
                one frame, JPEGs one after another (sizes in the header); {"frame", "cam", "sent"} = one camera.
                Reply (JSON): {"type": "detections", "frame", "sent", "infer_ms", "cams": {"1": {"players": [...],
                "ball": {...} | null}, ...}}. The cameras of one message run as ONE batch on the GPU.
Key: ROBOFLOW_API_KEY in the environment (never printed).
"""
from __future__ import annotations

import argparse
import asyncio
import json
import os
import struct
import time

import cv2
import numpy as np
from fastapi import FastAPI, WebSocket, WebSocketDisconnect

MODEL_ID = "football-players-detection-3zvbc/11"
PLAYER_CLASSES = {"player", "goalkeeper", "referee"}
ASPECT = 1.7778                      # src/detect: model_aspect
PLAYER_CONFIDENCE = 0.3              # config.yaml detection.player_confidence
BALL_CONFIDENCE = 0.3                # config.yaml detection.ball_confidence


def pad_to_aspect(frame: np.ndarray, aspect: float) -> tuple[np.ndarray, int]:
    """Copy of src/detect.pad_to_aspect: centre the frame on a black canvas `aspect` times wider than high."""
    h, w = frame.shape[:2]
    width = int(round(h * aspect))
    if w >= width:
        return frame, 0
    x = (width - w) // 2
    canvas = np.zeros((h, width, 3), dtype=frame.dtype)
    canvas[:, x:x + w] = frame
    return canvas, x


def _round(box, x_offset: int = 0) -> list[float]:
    x1, y1, x2, y2 = (float(v) for v in box)
    return [round(x1 - x_offset, 1), round(y1, 1), round(x2 - x_offset, 1), round(y2, 1)]


def parse(data: bytes) -> tuple[dict, list[tuple[int, bytes]]]:
    """One client message -> (header, [(cam, jpeg), ...])."""
    n = struct.unpack("<I", data[:4])[0]
    header = json.loads(data[4:4 + n])
    body = data[4 + n:]
    if "items" not in header:
        return header, [(int(header["cam"]), body)]
    out, k = [], 0
    for item in header["items"]:
        out.append((int(item["cam"]), body[k:k + int(item["size"])]))
        k += int(item["size"])
    return header, out


class FootballDetector:
    """Loads the Roboflow model once (GPU if onnxruntime has CUDA) and detects a batch of pictures."""

    def __init__(self):
        import onnxruntime as ort

        have = ort.get_available_providers()
        if "CUDAExecutionProvider" in have and hasattr(ort, "preload_dlls") and os.environ.get("DETECTOR_OWN_CUDA") == "1":
            # Colab: load the CUDA / cuDNN that came WITH this onnxruntime (pip nvidia-* packages in its own environment).
            # torch's newer cuDNN (9.24 on the first Colab run) crashed the first convolution: CUDNN_BACKEND_API_FAILED.
            ort.preload_dlls()
        print("onnxruntime", ort.__version__, "providers", have, flush=True)
        if "ONNXRUNTIME_EXECUTION_PROVIDERS" not in os.environ:      # GPU first (Colab), Apple chip (Mac), CPU
            wanted = [p for p in ("CUDAExecutionProvider", "CoreMLExecutionProvider") if p in have] + ["CPUExecutionProvider"]
            os.environ["ONNXRUNTIME_EXECUTION_PROVIDERS"] = "[" + ",".join(wanted) + "]"
        import supervision as sv
        from inference import get_model

        self.sv = sv
        self.model = get_model(MODEL_ID, api_key=os.environ["ROBOFLOW_API_KEY"])
        self.ms: list[float] = []

    def providers(self) -> list[str]:
        try:
            import onnxruntime as ort

            return ort.get_available_providers()
        except Exception:                            # noqa: BLE001 — informative only
            return []

    def detect_many(self, frames: list[np.ndarray]) -> list[dict]:
        padded = [pad_to_aspect(f, ASPECT) for f in frames]
        start = time.time()
        results = self.model.infer([p for p, _ in padded], confidence=min(PLAYER_CONFIDENCE, BALL_CONFIDENCE))
        self.ms.append((time.time() - start) * 1000 / max(1, len(frames)))
        self.ms = self.ms[-50:]
        return [self._one(self.sv.Detections.from_inference(r), x) for r, (_, x) in zip(results, padded)]

    def _one(self, dets, offset: int) -> dict:
        names = dets.data.get("class_name", np.array([]))
        players = [{"box": _round(b, offset), "cls": str(n), "conf": round(float(c), 3)}
                   for b, c, n in zip(dets.xyxy, dets.confidence, names)
                   if n in PLAYER_CLASSES and c >= PLAYER_CONFIDENCE]
        balls = [(b, c) for b, c, n in zip(dets.xyxy, dets.confidence, names) if n == "ball" and c >= BALL_CONFIDENCE]
        ball = None
        if balls:
            box, conf = max(balls, key=lambda bc: bc[1])
            ball = {"box": _round(box, offset), "conf": round(float(conf), 3)}
        return {"players": players, "ball": ball}

    def warm_up(self, n: int = 10, size: tuple[int, int] = (1280, 960), cams: int = 3) -> float:
        """ms per camera frame on random pictures (batches of `cams`)."""
        frame = np.random.randint(0, 255, (size[1], size[0], 3), np.uint8)
        self.detect_many([frame] * cams)            # first call compiles / allocates
        self.ms.clear()
        for _ in range(n):
            self.detect_many([frame] * cams)
        return float(np.mean(self.ms))

    def health(self) -> dict:
        return {"model": MODEL_ID, "ready": True, "ms_per_frame": round(float(np.mean(self.ms)), 1) if self.ms else None,
                "providers": self.providers()}


def make_app(detector: FootballDetector) -> FastAPI:
    app = FastAPI(title="football detector")

    @app.get("/health")
    def health() -> dict:
        return detector.health()

    @app.websocket("/detect")
    async def detect(ws: WebSocket) -> None:
        await ws.accept()
        loop = asyncio.get_running_loop()
        try:
            while True:
                data = await ws.receive_bytes()
                header, items = parse(data)
                frames = [cv2.imdecode(np.frombuffer(j, np.uint8), cv2.IMREAD_COLOR) for _, j in items]
                start = time.time()
                found = await loop.run_in_executor(None, detector.detect_many, frames)
                await ws.send_json({"type": "detections", "frame": header["frame"], "sent": header.get("sent"),
                                    "infer_ms": round((time.time() - start) * 1000, 1),
                                    "cams": {str(c): r for (c, _), r in zip(items, found)}})
        except WebSocketDisconnect:
            pass

    return app


if __name__ == "__main__":
    parser = argparse.ArgumentParser()
    parser.add_argument("--port", type=int, default=8766)
    args = parser.parse_args()
    import uvicorn

    detector = FootballDetector()
    print(f"detector ready: {detector.warm_up(5):.0f} ms per camera frame | providers {detector.providers()}", flush=True)
    uvicorn.run(make_app(detector), host="127.0.0.1", port=args.port, log_level="warning")


In [ ]:
import glob, os, subprocess, site, time, requests
try:
    from google.colab import userdata
    roboflow_key = userdata.get('ROBOFLOW_API_KEY')      # never printed
except Exception:
    roboflow_key = None
if not roboflow_key:
    print('No ROBOFLOW_API_KEY secret (name it exactly so, Notebook access on): the detector stays off.')
else:
    # Try 1: the CUDA/cuDNN that came with the detector's own onnxruntime. Try 2: torch's GPU libraries.
    torch_libs = [d for sp in site.getsitepackages() for d in glob.glob(sp + '/nvidia/*/lib')]
    h = None
    for attempt, (own, libs) in enumerate([('1', []), ('0', torch_libs)], 1):
        env = dict(os.environ, ROBOFLOW_API_KEY=roboflow_key, DETECTOR_OWN_CUDA=own)
        env['LD_LIBRARY_PATH'] = ':'.join(libs + [os.environ.get('LD_LIBRARY_PATH', '')])
        detector_proc = subprocess.Popen(['/content/detenv/bin/python', 'detector_server.py', '--port', '8766'],
                                         env=env, stdout=open(f'detector{attempt}.log', 'w'), stderr=subprocess.STDOUT)
        for _ in range(150):                                # model download + GPU warm-up: up to ~5 min
            time.sleep(2)
            try:
                h = requests.get('http://127.0.0.1:8766/health', timeout=2).json(); break
            except Exception:
                if detector_proc.poll() is not None: break
        if h and detector_proc.poll() is None:
            print(f'detector ready (try {attempt}):', h); break
        print(f'detector try {attempt} FAILED. Last lines of detector{attempt}.log:')
        print(''.join(open(f'detector{attempt}.log').readlines()[-12:]))
        detector_proc.kill(); h = None
    if h:
        if 'CUDAExecutionProvider' not in h.get('providers', []):
            print('WARNING: no CUDA for onnxruntime: the detector runs on the CPU (too slow for live). See detector.log')

## 3. Load the model, warm it up, and self-test
Downloads `config.yaml` + `model.pt` (150 MB) from Hugging Face. The self-test runs the model on fake frames and checks the speed: the **real-time factor** (inference time ÷ 2 s stride) must be < 1.

In [ ]:
import time, numpy as np, cv2
from specialist_server import Spotter, Session, make_app
spotter = Spotter()
print(spotter.health())
print('warm-up (first run is slow):', round(spotter.warm_up(), 2), 's')
times = [spotter.warm_up() for _ in range(3)]
print(f'inference on a {spotter.clip_len}-frame window: {np.mean(times)*1000:.0f} ms '
      f'-> real-time factor {np.mean(times)/2.0:.3f} (must be < 1)')
# Self-test of the live session logic with synthetic JPEG frames
session = Session(spotter, {'min_confidence': 0.3, 'stride_s': 2.0})
ok, jpg = cv2.imencode('.jpg', np.random.randint(0, 255, (224, 398, 3), np.uint8))
for i in range(30):
    due = session.add(i / spotter.fps, jpg.tobytes())
result = session.infer()
print('self-test OK:', {k: result[k] for k in ('frames', 'infer_ms', 'rtf', 'window')})

## 4. Start the server (in the background)

In [ ]:
import threading, uvicorn, requests
PORT = 8765
server = uvicorn.Server(uvicorn.Config(make_app(spotter), host='0.0.0.0', port=PORT, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
time.sleep(3)
print('local /health:', requests.get(f'http://127.0.0.1:{PORT}/health').json()['model'])

## 4b. Detector self-test (through the main server, like the Mac will use it)
30 frames x 3 cameras at 1280x960, sent as the Mac sends them. Live needs 15 camera frames per second, so the GPU time must be **well under 66 ms per camera frame**.

In [ ]:
import json, struct, asyncio, numpy as np, cv2, websockets
async def detector_selftest(n=30, cams=3):
    async with websockets.connect('ws://127.0.0.1:8765/detect', max_size=2**24) as ws:
        ok, jpg = cv2.imencode('.jpg', np.random.randint(0, 255, (960, 1280, 3), np.uint8), [cv2.IMWRITE_JPEG_QUALITY, 85])
        jpg = jpg.tobytes(); gpu = []; t0 = time.time()
        for i in range(n):
            head = json.dumps({'frame': i, 'sent': time.time(), 'items': [{'cam': c, 'size': len(jpg)} for c in range(1, cams + 1)]}).encode()
            await ws.send(struct.pack('<I', len(head)) + head + jpg * cams)
            r = json.loads(await ws.recv())
            if r.get('type') == 'error': print(r); return
            gpu.append(r['infer_ms'] / cams)
        wall = (time.time() - t0) * 1000 / (n * cams)
        print(f'detector: {np.mean(gpu[3:]):.0f} ms per camera frame on the GPU, {wall:.0f} ms including JPEG decode '
              f'-> {"OK for live" if wall < 66 else "TOO SLOW for 15 frames/s"}')
if roboflow_key: await detector_selftest()
else: print('detector off (no ROBOFLOW_API_KEY secret)')

## 5. Open a public tunnel and print the URL
Default: a free Cloudflare quick tunnel (no account, no data cap). The detector sends ~19 Mbit/s (~7 GB per half), far over ngrok's free 1 GB/month: set `TUNNEL = 'ngrok'` only for the action model alone (it uses the `NGROK_AUTHTOKEN` secret).

In [ ]:
import re, subprocess
from IPython.display import HTML, display
TUNNEL = 'cloudflare'   # or 'ngrok'
public_url = None
try:
    from google.colab import userdata
    token = userdata.get('NGROK_AUTHTOKEN') if TUNNEL == 'ngrok' else None
except Exception:
    token = None
if token:
    from pyngrok import ngrok, conf
    conf.get_default().auth_token = token
    public_url = ngrok.connect(PORT, 'http').public_url.replace('http://', 'https://')
else:
    print('Cloudflare quick tunnel (no account needed).')
    !wget -q -O cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x cloudflared
    tunnel = subprocess.Popen(['./cloudflared', 'tunnel', '--no-autoupdate', '--url', f'http://127.0.0.1:{PORT}'],
                              stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in tunnel.stdout:
        found = re.search(r'https://[a-z0-9-]+\.trycloudflare\.com', line)
        if found:
            public_url = found.group(0)
            break
    threading.Thread(target=lambda: [None for _ in tunnel.stdout], daemon=True).start()  # keep reading its log
    time.sleep(5)
for attempt in range(10):   # check /health THROUGH the tunnel
    try:
        r = requests.get(public_url + '/health', headers={'ngrok-skip-browser-warning': '1'}, timeout=10)
        print('/health through the tunnel:', r.status_code, r.json()['gpu']); break
    except Exception as error:
        print('waiting for the tunnel...', type(error).__name__); time.sleep(3)
display(HTML(f'<div style="font-size:28px;font-weight:bold;padding:16px;border:3px solid #22c55e;'
             f'border-radius:8px">Paste this into the dashboard:<br><code>{public_url}</code></div>'))
print(public_url)

## 6. Keep this running
This cell prints a heartbeat every minute so you can see the server is alive. Stop it with the ■ button when you are done.

In [ ]:
while True:
    print(time.strftime('%H:%M:%S'), 'server alive at', public_url)
    time.sleep(60)